# PyTorch Training Loop

A training loop repeatedly trains a neural network on the
training dataset.

The basic process is:

1. Get input data
2. Forward pass
3. Calculate prediction
4. Calculate loss
5. Clear old gradients
6. Backpropagation
7. Update weights
8. Repeat

The core PyTorch training loop is:

optimizer.zero_grad()

output = model(X)

loss = loss_function(output, y)

loss.backward()

optimizer.step()

In [1]:
import torch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Create dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=7,
    n_redundant=3,
    n_classes=2,
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Convert NumPy Data to PyTorch Tensors

PyTorch models work with PyTorch tensors.

Therefore, we convert:

X_train → torch.float32
y_train → torch.float32

For binary classification, the target should have a shape
compatible with the model output.

In [2]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([800, 10])
y_train: torch.Size([800, 1])
X_test: torch.Size([200, 10])
y_test: torch.Size([200, 1])


In [3]:
import torch.nn as nn

class ANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(10, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)
        x = self.sigmoid(x)

        return x


model = ANN()

print(model)

ANN(
  (fc1): Linear(in_features=10, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


## Loss Function

Our problem is binary classification.

Therefore, we use Binary Cross Entropy Loss.

PyTorch provides:

nn.BCELoss()

The loss measures how different the model's predictions
are from the actual labels.

In [4]:
loss_function = nn.BCELoss()

## Optimizer

The optimizer updates the model's weights and biases
using the gradients calculated during backpropagation.

We will use the Adam optimizer.

PyTorch provides:

torch.optim.Adam

In [5]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [6]:
# Forward pass
output = model(X_train_tensor)

# Calculate loss
loss = loss_function(
    output,
    y_train_tensor
)

print("Loss:", loss.item())

Loss: 0.7167650461196899


In [7]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

# Complete Training Loop

An epoch means the model goes through the entire training dataset.

We repeat the training process for multiple epochs.

For every epoch:

1. Forward pass
2. Calculate loss
3. Clear gradients
4. Backpropagation
5. Update parameters
6. Print loss

In [8]:
epochs = 50

for epoch in range(epochs):

    # Forward pass
    output = model(X_train_tensor)

    # Calculate loss
    loss = loss_function(
        output,
        y_train_tensor
    )

    # Clear previous gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [5/50], Loss: 0.6684
Epoch [10/50], Loss: 0.6270
Epoch [15/50], Loss: 0.5892
Epoch [20/50], Loss: 0.5527
Epoch [25/50], Loss: 0.5173
Epoch [30/50], Loss: 0.4826
Epoch [35/50], Loss: 0.4485
Epoch [40/50], Loss: 0.4160
Epoch [45/50], Loss: 0.3860
Epoch [50/50], Loss: 0.3590


## What is an Epoch?

One epoch means the model has processed the complete
training dataset once.

For example:

800 training samples

epochs = 10

means the model sees those 800 samples 10 times.

More epochs do not always mean a better model.

Too many epochs can cause overfitting.

## Batch Training

Instead of giving all 800 samples to the model at once,
we can divide them into smaller batches.

For example:

800 samples
batch size = 32

Number of batches:

800 / 32 = 25 batches

The model processes:

Batch 1 → 32 samples
Batch 2 → 32 samples
Batch 3 → 32 samples
...
Batch 25 → 32 samples

After all batches are processed,
one epoch is complete.

In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [10]:
for X_batch, y_batch in train_loader:

    print("X batch:", X_batch.shape)
    print("y batch:", y_batch.shape)

    break

X batch: torch.Size([32, 10])
y batch: torch.Size([32, 1])


In [11]:
epochs = 50

for epoch in range(epochs):

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # Forward pass
        output = model(X_batch)

        # Calculate loss
        loss = loss_function(
            output,
            y_batch
        )

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {average_loss:.4f}"
        )

Epoch [5/50], Loss: 0.2076
Epoch [10/50], Loss: 0.1646
Epoch [15/50], Loss: 0.1432
Epoch [20/50], Loss: 0.1254
Epoch [25/50], Loss: 0.1112
Epoch [30/50], Loss: 0.1039
Epoch [35/50], Loss: 0.0977
Epoch [40/50], Loss: 0.0835
Epoch [45/50], Loss: 0.0783
Epoch [50/50], Loss: 0.0705


## shuffle=True

shuffle=True randomly changes the order of training samples
at the beginning of each epoch.

This helps prevent the model from learning unwanted
patterns caused by the original ordering of the dataset.

## Model Evaluation

During evaluation:

- We don't need gradients.
- We don't update weights.
- We only make predictions.

PyTorch provides:

torch.no_grad()

to disable gradient tracking.

In [12]:
model.eval()

with torch.no_grad():

    predictions = model(X_test_tensor)

print(predictions[:10])

tensor([[1.0000e+00],
        [2.9850e-02],
        [3.3888e-04],
        [1.2458e-03],
        [1.0000e+00],
        [9.9998e-01],
        [8.2742e-02],
        [9.9997e-01],
        [1.7277e-05],
        [3.8595e-03]])


In [13]:
predicted_classes = (
    predictions >= 0.5
).int()

print(predicted_classes[:20])

tensor([[1],
        [0],
        [0],
        [0],
        [1],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1],
        [1],
        [1],
        [0],
        [0],
        [0],
        [1],
        [0],
        [1],
        [0]], dtype=torch.int32)


In [14]:
correct = (
    predicted_classes == y_test_tensor
).sum().item()

total = y_test_tensor.size(0)

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.925


In [15]:
epochs = 50

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # 1. Forward pass
        output = model(X_batch)

        # 2. Calculate loss
        loss = loss_function(output, y_batch)

        # 3. Clear old gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/50], Loss: 0.0661
Epoch [2/50], Loss: 0.0619
Epoch [3/50], Loss: 0.0641
Epoch [4/50], Loss: 0.0648
Epoch [5/50], Loss: 0.0612
Epoch [6/50], Loss: 0.0565
Epoch [7/50], Loss: 0.0549
Epoch [8/50], Loss: 0.0513
Epoch [9/50], Loss: 0.0514
Epoch [10/50], Loss: 0.0522
Epoch [11/50], Loss: 0.0501
Epoch [12/50], Loss: 0.0481
Epoch [13/50], Loss: 0.0492
Epoch [14/50], Loss: 0.0457
Epoch [15/50], Loss: 0.0437
Epoch [16/50], Loss: 0.0415
Epoch [17/50], Loss: 0.0414
Epoch [18/50], Loss: 0.0397
Epoch [19/50], Loss: 0.0388
Epoch [20/50], Loss: 0.0370
Epoch [21/50], Loss: 0.0374
Epoch [22/50], Loss: 0.0354
Epoch [23/50], Loss: 0.0353
Epoch [24/50], Loss: 0.0334
Epoch [25/50], Loss: 0.0340
Epoch [26/50], Loss: 0.0325
Epoch [27/50], Loss: 0.0302
Epoch [28/50], Loss: 0.0290
Epoch [29/50], Loss: 0.0292
Epoch [30/50], Loss: 0.0288
Epoch [31/50], Loss: 0.0303
Epoch [32/50], Loss: 0.0263
Epoch [33/50], Loss: 0.0250
Epoch [34/50], Loss: 0.0250
Epoch [35/50], Loss: 0.0235
Epoch [36/50], Loss: 0.0227
E